# Degrau de Potencial — Controle Difusional

Considera-se uma espécie eletroativa $O$ que sofre redução irreversível na superfície do eletrodo:

$$O + ne^- \rightarrow R$$

Um degrau de potencial suficientemente grande é aplicado em $\tau > 0$, de modo que a concentração superficial de $O\rightarrow 0$ instantaneamente. O transporte de massa é descrito pela segunda lei de Fick:

$$\frac{\partial C_O}{\partial \tau} = D_O \frac{\partial^2 C_O}{\partial y^2} \tag{1.0}$$

com condição inicial e condições de contorno:

$$C_O(y, 0) = C_O^* \tag{1.1}$$

$$\lim_{y \to \infty} C_O(y, \tau) = C_O^* \tag{1.2}$$

$$C_O(0, \tau) = 0 \quad \text{para} \quad \tau > 0 \tag{1.3}$$

Introduzindo as variáveis adimensionais:

$$x = \frac{y}{\sqrt{D_O \tau_s}}, \quad t = \frac{\tau}{\tau_s}, \quad c = \frac{C_O}{C_O^*}$$

onde $\tau_s$ é uma escala de tempo de referência, as equações tornam-se:

$$\frac{\partial c}{\partial t} = \frac{\partial^2 c}{\partial x^2} \quad \text{para} \quad t \geq 0 \quad \text{e} \quad 0 \leq x \leq 6 \tag{1.0}$$

$$c(x, 0) = 1 \tag{1.1}$$

$$\lim_{x \to \infty} c(x, t) = 1 \tag{1.2}$$

$$c(0, t) = 0 \quad \text{para} \quad t > 0 \tag{1.3}$$

A solução analítica para o perfil de concentração é:

$$c(x, t) = \text{erf}\left(\frac{x}{2\sqrt{t}}\right) \tag{2.0}$$

A densidade de corrente é proporcional ao fluxo difusional na superfície do eletrodo:

$$j(\tau) = nFD_O \left(\frac{\partial C_O}{\partial y}\right)_{y=0}$$

Definindo a corrente adimensional:

$$i(t) = \frac{j(\tau) \sqrt{D_O \tau_s}}{nFD_O C_O^*}$$

e derivando a expressão analítica da concentração, obtém-se a **equação de Cottrell**:

$$i(t) = \frac{1}{\sqrt{\pi t}} \tag{3.0}$$

## Bibliotecas:

In [ ]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from scipy import special
from ipywidgets import Text, Button, HBox, VBox, Output
from IPython.display import display, clear_output

## Definindo o Modelo:

In [ ]:

model = pybamm.BaseModel()

concentration = pybamm.Variable("Concentração", domain="electrolyte")

flux = -pybamm.grad(concentration)        # fluxo difusional
dcdt = -pybamm.div(flux)                  # lei de Fick (eq. 1.0)

model.rhs = {concentration: dcdt}

# condição inicial (eq. 1.1): concentração uniforme adimensionalizada
model.initial_conditions = {concentration: pybamm.Scalar(1)}

# condições de contorno — Dirichlet
# esquerda (x=0): superfície do eletrodo — condição de Cottrell (eq. 1.3)
# direita  (x=6): seio da solução        — difusão semi-infinita  (eq. 1.2)
model.boundary_conditions = {
    concentration: {
        "left":  (pybamm.Scalar(0), "Dirichlet"),
        "right": (pybamm.Scalar(1), "Dirichlet"),
    }
}

model.variables = {
    "Concentração": concentration,
    "Fluxo":        flux,
}


# ── Geometria e Malha ─────────────────────────────────────────────────────────

x_variable = pybamm.SpatialVariable(
    "x", domain=["electrolyte"], coord_sys="cartesian"
)

geometry = {
    "electrolyte": {x_variable: {"min": pybamm.Scalar(0), "max": pybamm.Scalar(6)}}
}

submesh_types   = {
    "electrolyte": pybamm.MeshGenerator(
        pybamm.Exponential1DSubMesh,
        submesh_params={
            "side": "left",
            "stretch": 5,
        },
    )
}
variable_points  = {x_variable: 400}
mesh             = pybamm.Mesh(geometry, submesh_types, variable_points)


# ── Discretização ─────────────────────────────────────────────────────────────

spatial_methods = {"electrolyte": pybamm.FiniteVolume()}
discretisation  = pybamm.Discretisation(mesh, spatial_methods)
discretisation.process_model(model)

## Solução Numérica:

In [ ]:
# o modelo de Cottrell não tem parâmetros livres: resolve-se uma única vez
solver   = pybamm.ScipySolver()
time     = np.linspace(1e-5, 1, 1000)
solution = solver.solve(model, time)

concentration_solution = solution["Concentração"]   # interpolável em (x, t)
flux_solution          = solution["Fluxo"]           # interpolável em (x, t)

## Funções Analíticas:

In [ ]:
def analytical_concentration(x, t):
    """Concentração analítica: c(x,t) = erf(x / 2√t)"""
    return special.erf(x / (2 * np.sqrt(t)))

def analytical_current(t):
    """Corrente analítica adimensional: i(t) = −1/√(πt)
    Negativa por convenção catódica (redução).
    """
    return -1 / np.sqrt(np.pi * t)

## Gráfico Estático:

In [ ]:
x_plot = np.linspace(0, 6, 200)
times  = [1, 0.1, 0.01, 0.001]
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

for t_i, color in zip(times, colors):
    # numérico — pontos
    ax2.plot(
        x_plot,
        concentration_solution(t=t_i, x=x_plot),
        ".",
        color=color,
        markersize=8,
        label=f"t = {t_i}",
    )
    # analítico sem label — linha de referência, não entra na legenda
    ax2.plot(
        x_plot,
        analytical_concentration(x_plot, t_i),
        "-",
        color=color,
        linewidth=2,
    )

ax2.set_xlabel(r"$x$")
ax2.set_ylabel(r"$c$")
ax2.set_xlim([0, 0.9])
ax2.set_ylim([0, 1.05])
ax2.set_title("Perfis de concentração — Potencial de Cottrell")
ax2.legend(fontsize=8, ncol=2)

ax1.plot(solution.t, flux_solution(solution.t, x=0), "r.", label="Numérico")
ax1.plot(solution.t, analytical_current(solution.t),  "b-", label="Analítico")
ax1.set_xlabel(r"$t$")
ax1.set_ylabel(r"$i$")
ax1.set_xlim([0.01, 1])
ax1.set_ylim([-10, 0])
ax1.legend()

plt.tight_layout()
plt.show()

O gráfico da esquerda mostra a corrente em função do tempo. A corrente decai monotonicamente com $t^{-1/2}$, comportamento característico da **equação de Cottrell**: quanto maior o tempo, mais espessa é a camada de difusão, menor é o gradiente de concentração na superfície e, portanto, menor é a corrente.

O gráfico da direita mostra os perfis de concentração de $O$ em quatro instantes de tempo distintos. Os pontos representam a solução numérica e as linhas a solução analítica (eq. 2.0). O acordo entre as duas confirma a validade da discretização. À medida que o tempo avança, a camada de difusão se estende pelo domínio — a concentração superficial permanece nula (condição de Cottrell) enquanto a perturbação se propaga progressivamente para o seio da solução.

## Gráfico Interativo:

In [ ]:
x_plot = np.linspace(0, 6, 200)
output = Output()

def plot(time_1, time_2):
    with output:
        clear_output(wait=True)

        # validação das entradas
        try:
            t1 = float(time_1)
            t2 = float(time_2)
        except ValueError:
            print("Por favor, insira valores numéricos válidos.")
            return

        if not (1e-5 <= t1 <= 1) or not (1e-5 <= t2 <= 1):
            print("Os tempos devem estar entre 1×10⁻⁵ e 1.")
            return

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

        for t_i, color, label in [
            (t1, "tab:blue", "1° Tempo"),
            (t2, "tab:red",  "2° Tempo"),
        ]:
            ax2.plot(
                x_plot,
                concentration_solution(t=t_i, x=x_plot),
                ".",
                color=color,
                markersize=10,
                label=f"{label} (t = {t_i})",
            )
            # analítico sem label — linha de referência, não entra na legenda
            ax2.plot(
                x_plot,
                analytical_concentration(x_plot, t_i),
                "-",
                color=color,
                linewidth=2,
            )

        ax2.set_xlabel(r"$x$")
        ax2.set_ylabel(r"$c$")
        ax2.set_xlim([0, 1])
        ax2.set_ylim([0, 1.05])
        ax2.set_title("Perfis de concentração — Potencial de Cottrell")
        ax2.legend(fontsize=8)

        ax1.plot(solution.t, flux_solution(solution.t, x=0), "r.", label="Numérico")
        ax1.plot(solution.t, analytical_current(solution.t),  "b-", label="Analítico")
        ax1.set_xlabel(r"$t$")
        ax1.set_ylabel(r"$i$")
        ax1.set_xlim([0.01, 1])
        ax1.set_ylim([-10, 0])
        ax1.legend()

        plt.tight_layout()
        display(fig)
        plt.close(fig)


# ── Widgets ───────────────────────────────────────────────────────────────────

field_t1 = Text(
    value="0.1",
    description="1° Tempo:",
    style={"description_width": "initial"},
)

field_t2 = Text(
    value="0.01",
    description="2° Tempo:",
    style={"description_width": "initial"},
)

button = Button(description="Recalcular", button_style="success")

def on_click(b):
    plot(field_t1.value, field_t2.value)

button.on_click(on_click)

interface = VBox([HBox([field_t1, field_t2, button]), output])
display(interface)

# exibe o gráfico inicial com os valores padrão
plot(field_t1.value, field_t2.value)